In [ ]:
import vr1
from vr1.core import FuelAssembly, Lattice
from vr1.settings import VR1Settings
from vr1.writer import WriterOpenMC
from vr1.plots import test_plots
from vr1.materials import VR1Materials
from vr1.VR1facility import Facility
import vr1.lattice_units as vlu
import openmc
from vr1.core import core_designs
import vr1.utils
import pke.solver as vr1pke

# Reactor Kinetics

The vr1-openmc package comes with a point kinetics equations (PKE) solver. You can play with this below. The function *rho_t* is passed to the solver as the reactivity within the system. You can make this return whatever you like to see what would happen to the neutron density (AKA power) and the precursor concentration. The units of reactivity are Δk/k, which is equivalent to pcm. 

In [ ]:
def rho_t(t):
    return 0

testpke = vr1pke.PointKineticsEquationSolver(rho_t)
testpke.solve(t_span = (0,1000))
testpke.plot_neutron_density()
testpke.plot_precursors()

# Kinetics Parameters from OpenMC

You are able to obtain kinetics parameters, such as $\beta_{eff}$ from OpenMC directly by using Tallies. Monte Carlo codes do not provide every possible piece of information to you by default, that would be much too expensive. Most of the interesting information that you can get must be specified before you run the code. 
Examples of tally types include:

* Heating
* Flux 
* Reaction Rates
* and more!

To obtain kinetics parameters, you have to initialize some of these tallies. This is done for you in the cell below. Run the code for a core design of your choice to obtain the effective delayed neutron fraction.

In [ ]:
import os


vr1_materials = VR1Materials()
mats = vr1_materials.get_materials()
# mats.export_to_xml()


lattice_input =   list

lattice_obj = Lattice(materials=vr1_materials,lattice_str=lattice_input)

facility_obj = Facility(materials=vr1_materials)
vr1_model = facility_obj.build(lattice=lattice_obj)

geometry = openmc.Geometry(root=vr1_model)

batches = 40


settings = openmc.Settings()
settings.run_mode = 'eigenvalue'
settings.temperature = {'method':'interpolation','range':(293.15,923.15)} #unsure
settings.batches = batches
settings.inactive = 15 
settings.particles = 10000
settings.photon_transport = True
source_area = openmc.stats.Box(lattice_obj.source_lower_left,lattice_obj.source_upper_right)
settings.source_rejection_fraction = 0.01
settings.source = openmc.Source(space=source_area,constraints={'fissionable': True})


tally = openmc.Tally(name="ifp-scores")
tally.scores = [
    "ifp-time-numerator",
    "ifp-beta-numerator",
    "ifp-denominator"
]

tallies =  openmc.Tallies([tally])

model = openmc.Model(geometry=geometry,materials=mats,settings=settings,tallies=tallies)
model.export_to_model_xml()

openmc.run()

with openmc.StatePoint(f'statepoint.{batches}.h5') as sp:
    k_eff = sp.keff.nominal_value
    generation_time, beta_eff = sp.get_kinetics_parameters()


print(beta_eff)
